# Multi-GPU Alignment: SFT → DPO → RLHF

The full alignment pipeline on a tiny model, with model parallelism.

- **Model:** Qwen2.5-0.5B (latest, tiny, fits anywhere)
- **LoRA** on all three stages
- **DeepSpeed ZeRO-3** — model parallelism (shards weights across GPUs)
- **Platform:** Kaggle 2x T4 (free)

### The 3-stage alignment pipeline

```
Stage 1: SFT          "Learn to follow instructions"
  |
Stage 2: DPO          "Learn which answers humans prefer" (offline)
  |
Stage 3: RLHF (PPO)   "Optimize a reward signal live" (online)
```

### ZeRO-3 vs ZeRO-2 (model parallelism)

```
ZeRO-2: splits optimizer + gradients across GPUs
         model weights stay on every GPU (duplicated)

ZeRO-3: splits optimizer + gradients + MODEL WEIGHTS across GPUs
         each GPU holds only a shard — fits bigger models
         = model parallelism, zero code changes
```

## Step 1: Install

In [ ]:
!pip install -q transformers datasets peft accelerate "trl>=0.12" deepspeed

## Step 2: Detect GPUs

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

## Step 3: Configs — DeepSpeed ZeRO-3

ZeRO-3 = model parallelism without any code changes.
It shards model weights, optimizer states, AND gradients across GPUs.

| What | ZeRO-2 | ZeRO-3 |
|------|--------|--------|
| Optimizer | Sharded | Sharded |
| Gradients | Sharded | Sharded |
| **Model weights** | **Duplicated** | **Sharded** |
| Max model size | Limited by 1 GPU | Limited by ALL GPUs combined |

In [ ]:
# Accelerate config with ZeRO-3 (model parallelism)
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: true
  zero3_save_16bit_model: true
  zero_stage: 3
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Config: {NUM_GPUS} GPU(s), ZeRO-3 (model parallelism), bf16")
print(f"Model weights sharded across {NUM_GPUS} GPU(s) + CPU offload")

---
# Part 1: SFT — Teach It to Follow Instructions

**Input:** instruction → **Output:** helpful response

We fine-tune on 500 Alpaca examples so the model learns the
instruction-following format.

In [ ]:
%%writefile train_sft.py
"""Stage 1: SFT — supervised fine-tuning on instructions."""
import torch, os
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

MODEL = "Qwen/Qwen2.5-0.5B"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True)

# Data: 500 Alpaca instruction-output pairs
dataset = load_dataset("tatsu-lab/alpaca", split="train")
dataset = dataset.shuffle(seed=42).select(range(500))

def format_example(ex):
    prompt = ex["instruction"]
    if ex.get("input"):
        prompt += f"\n{ex['input']}"
    return {"text": f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{ex['output']}<|im_end|>"}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(f"SFT dataset: {len(dataset)} examples")

# Train with LoRA
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="./sft_output",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=5,
        bf16=True,
        gradient_checkpointing=True,
        max_seq_length=512,
        dataset_text_field="text",
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    ),
)

trainer.train()
trainer.save_model("./sft_output/final")
tokenizer.save_pretrained("./sft_output/final")

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    print("\nSFT complete!")

In [ ]:
print(f"=== Stage 1: SFT on {NUM_GPUS} GPU(s) ===")
start = time.time()
!accelerate launch --num_processes={NUM_GPUS} train_sft.py
sft_time = time.time() - start
print(f"SFT done! {sft_time:.0f}s")

---
# Part 2: DPO — Learn Human Preferences (Offline)

**Input:** prompt + chosen response + rejected response

DPO directly optimizes the model to prefer the "chosen" answer
over the "rejected" one. No reward model needed.

```
Prompt: "What is gravity?"
Chosen:   "Gravity is the force that attracts objects with mass..." ✓
Rejected: "Gravity is a thing. It makes stuff fall."               ✗
                                                                    
DPO loss: make P(chosen) > P(rejected)
```

In [ ]:
%%writefile train_dpo.py
"""Stage 2: DPO — direct preference optimization."""
import torch, os
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig

MODEL = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True)

# Load preference dataset: each row has prompt + chosen + rejected
dataset = load_dataset("argilla/ultrafeedback-binarized-preferences-cleaned", split="train")
dataset = dataset.shuffle(seed=42).select(range(500))

# Format into chat messages
def format_dpo(ex):
    prompt_text = ex["chosen"][0]["content"] if ex["chosen"] else ""
    chosen_text = ex["chosen"][1]["content"] if len(ex["chosen"]) > 1 else ""
    rejected_text = ex["rejected"][1]["content"] if len(ex["rejected"]) > 1 else ""
    return {
        "prompt": [{"role": "user", "content": prompt_text}],
        "chosen": [
            {"role": "user", "content": prompt_text},
            {"role": "assistant", "content": chosen_text},
        ],
        "rejected": [
            {"role": "user", "content": prompt_text},
            {"role": "assistant", "content": rejected_text},
        ],
    }

dataset = dataset.map(format_dpo, remove_columns=dataset.column_names)
print(f"DPO dataset: {len(dataset)} preference pairs")

# Train
trainer = DPOTrainer(
    model=model,
    args=DPOConfig(
        output_dir="./dpo_output",
        beta=0.1,                    # How much to trust preferences (lower = stronger)
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        warmup_steps=10,
        logging_steps=5,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
        max_length=512,
        max_prompt_length=256,
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    ),
)

trainer.train()
trainer.save_model("./dpo_output/final")
tokenizer.save_pretrained("./dpo_output/final")

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    print("\nDPO complete!")

In [ ]:
print(f"=== Stage 2: DPO on {NUM_GPUS} GPU(s) ===")
start = time.time()
!accelerate launch --num_processes={NUM_GPUS} train_dpo.py
dpo_time = time.time() - start
print(f"DPO done! {dpo_time:.0f}s")

---
# Part 3: RLHF (PPO) — Optimize a Reward Signal (Online)

Unlike DPO (offline, fixed dataset), PPO generates responses live
and optimizes them against a reward model.

```
1. Model generates a response to a prompt
2. Reward model scores the response (0 = bad, 1 = good)
3. PPO updates the model to get higher scores
4. KL penalty prevents the model from drifting too far from base
```

We use a sentiment classifier as a simple reward model:
positive sentiment = good, negative = bad.
In production, you'd use a real preference reward model.

In [ ]:
%%writefile train_ppo.py
"""Stage 3: RLHF (PPO) — reinforcement learning from human feedback."""
import torch, os
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from peft import LoraConfig

MODEL = "Qwen/Qwen2.5-0.5B"

# Load model with a value head (PPO needs this to estimate state values)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none", task_type="CAUSAL_LM",
)

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
    peft_config=lora_config,
)

# Reward model: sentiment classifier (positive = high reward)
# In production, this would be a model trained on human preferences
reward_pipe = pipeline("sentiment-analysis", model="lvwerra/distilbert-imdb", device="cpu")

def get_reward(texts):
    """Score responses: positive sentiment = reward 1.0, negative = -1.0"""
    results = reward_pipe(texts, truncation=True, max_length=256)
    return [torch.tensor(r["score"] if r["label"] == "POSITIVE" else -r["score"]) for r in results]

# Prompts for PPO training
prompts_data = load_dataset("tatsu-lab/alpaca", split="train")
prompts_data = prompts_data.shuffle(seed=42).select(range(200))
prompts = [ex["instruction"] for ex in prompts_data]
print(f"PPO: {len(prompts)} prompts")

# PPO config
ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=8,
    mini_batch_size=4,
    ppo_epochs=2,
    gradient_accumulation_steps=2,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    tokenizer=tokenizer,
)

# Training loop
print("Starting PPO training...")
for i in range(0, len(prompts), ppo_config.batch_size):
    batch_prompts = prompts[i:i + ppo_config.batch_size]
    if len(batch_prompts) < ppo_config.batch_size:
        break

    # 1. Tokenize prompts
    query_tensors = [tokenizer.encode(p, return_tensors="pt", truncation=True, max_length=128).squeeze() for p in batch_prompts]

    # 2. Generate responses
    response_tensors = ppo_trainer.generate(
        query_tensors, max_new_tokens=64, temperature=0.7, do_sample=True,
    )

    # 3. Decode responses
    responses = [tokenizer.decode(r.squeeze(), skip_special_tokens=True) for r in response_tensors]

    # 4. Get rewards from reward model
    rewards = get_reward(responses)

    # 5. PPO step — update model to maximize reward
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    batch_num = i // ppo_config.batch_size + 1
    avg_reward = sum(r.item() for r in rewards) / len(rewards)
    if int(os.environ.get("LOCAL_RANK", 0)) == 0:
        print(f"  Batch {batch_num} | Avg reward: {avg_reward:.3f}")

# Save
model.save_pretrained("./ppo_output/final")
tokenizer.save_pretrained("./ppo_output/final")

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    print("\nPPO/RLHF complete!")

In [ ]:
print(f"=== Stage 3: PPO/RLHF on {NUM_GPUS} GPU(s) ===")
start = time.time()
!accelerate launch --num_processes={NUM_GPUS} train_ppo.py
ppo_time = time.time() - start
print(f"PPO done! {ppo_time:.0f}s")

---
## Results

In [ ]:
from IPython.display import HTML, display

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 780px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 20px; text-align: center;">
      <div style="color: #3fb950; font-size: 13px; font-weight: 600;">Stage 1: SFT</div>
      <div style="color: #58a6ff; font-size: 28px; font-weight: 700; margin: 8px 0;">{sft_time:.0f}s</div>
      <div style="color: #8b949e; font-size: 11px;">Learn to follow instructions</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 20px; text-align: center;">
      <div style="color: #f0883e; font-size: 13px; font-weight: 600;">Stage 2: DPO</div>
      <div style="color: #58a6ff; font-size: 28px; font-weight: 700; margin: 8px 0;">{dpo_time:.0f}s</div>
      <div style="color: #8b949e; font-size: 11px;">Offline preference alignment</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 20px; text-align: center;">
      <div style="color: #d2a8ff; font-size: 13px; font-weight: 600;">Stage 3: PPO</div>
      <div style="color: #58a6ff; font-size: 28px; font-weight: 700; margin: 8px 0;">{ppo_time:.0f}s</div>
      <div style="color: #8b949e; font-size: 11px;">Online reward optimization</div>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
    <table style="width:100%; color: #c9d1d9; font-size: 13px; border-spacing: 0 8px;">
      <tr><td style="color:#8b949e;">Model</td><td style="text-align:right; font-weight:600;">Qwen2.5-0.5B</td></tr>
      <tr><td style="color:#8b949e;">GPUs</td><td style="text-align:right;">{NUM_GPUS}x {torch.cuda.get_device_name(0)}</td></tr>
      <tr><td style="color:#8b949e;">DeepSpeed</td><td style="text-align:right; color:#3fb950;">ZeRO-3 (model parallelism)</td></tr>
      <tr><td style="color:#8b949e;">LoRA</td><td style="text-align:right;">r=16, q_proj + v_proj</td></tr>
      <tr><td style="color:#8b949e;">Total time</td><td style="text-align:right; font-weight:600;">{sft_time + dpo_time + ppo_time:.0f}s</td></tr>
    </table>
  </div>
</div>
"""
display(HTML(html))

## Test: Compare Responses

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

MODEL = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

def load_and_generate(adapter_path, label, prompts):
    """Load base + LoRA adapter, generate responses."""
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True).to("cuda")
    if adapter_path:
        model = PeftModel.from_pretrained(base, adapter_path)
    else:
        model = base
    model.eval()

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    for p in prompts:
        text = f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"\nQ: {p}")
        print(f"A: {response.strip()[:300]}")
    del model, base
    torch.cuda.empty_cache()

test_prompts = [
    "What is gravity?",
    "How do I learn programming?",
    "Why is the sky blue?",
]

load_and_generate(None, "BASE MODEL (no fine-tuning)", test_prompts)
load_and_generate("./sft_output/final", "AFTER SFT", test_prompts)
load_and_generate("./dpo_output/final", "AFTER DPO", test_prompts)

---

## How It All Fits Together

### The 3-stage pipeline

```
Base Model (Qwen2.5-0.5B)
    |
    | Stage 1: SFT
    | Train on: instruction → response pairs
    | Loss: next-token prediction (cross-entropy)
    | Result: model follows instructions
    |
    | Stage 2: DPO
    | Train on: chosen vs rejected responses
    | Loss: log P(chosen) - log P(rejected)
    | Result: model prefers better answers
    |
    | Stage 3: PPO (RLHF)
    | Train on: generated responses scored by reward model
    | Loss: PPO objective (maximize reward, minimize KL divergence)
    | Result: model optimizes for quality
    v
Aligned Model
```

### DPO vs RLHF (PPO)

| | DPO | RLHF (PPO) |
|--|-----|------------|
| **Data** | Fixed preference pairs | Generates responses live |
| **Reward model** | Not needed (implicit) | Required (explicit) |
| **Stability** | Very stable | Can be unstable |
| **Compute** | Same as SFT | 2-4x more (generation + reward) |
| **When to use** | You have preference data | You have a reward signal |

### ZeRO-3 model parallelism

```
Without ZeRO-3 (each GPU holds full model):
  GPU 0: [full model] [optimizer] [gradients]
  GPU 1: [full model] [optimizer] [gradients]
  → Each GPU needs enough memory for everything

With ZeRO-3 (model sharded across GPUs):
  GPU 0: [model shard A] [optim shard A] [grad shard A]
  GPU 1: [model shard B] [optim shard B] [grad shard B]
  → Model can be 2x bigger (or more with CPU offload)
  → Parameters gathered on-the-fly when needed
```

The only config change from ZeRO-2:
```yaml
zero_stage: 3              # was 2
zero3_init_flag: true      # new
offload_param_device: cpu  # new — offload model weights to CPU too
```

Zero code changes in the training scripts.

| Platform | GPUs | Cost |
|----------|------|------|
| **Kaggle** | 2x T4 | Free (30h/week) |
| **Colab** | 1x T4 | Free |